# Aprendizado de Máquina — Lista prática 07

## Classificação e Classificadores Gaussianos

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Nesta lista a população é **inteiramente conhecida**: duas gaussianas em $\mathbb{R}^2$
com médias e covariâncias que nós escolhemos. Isso permite algo que num banco real
nunca é possível — calcular o **erro de Bayes**, o piso que nenhum classificador
consegue furar, e medir a que distância dele cada método chega.

> **o QDA é o modelo *correto* para esta população. A lista mede o que ele ganha
> por isso, e quanto de dado ele precisa para cobrar o prêmio.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots
from scipy.stats import multivariate_normal

import sklearn.model_selection as skm
from sklearn.datasets import load_breast_cancer
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                           QuadraticDiscriminantAnalysis)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — a população e o erro de Bayes

Duas classes gaussianas em $\mathbb{R}^2$, com **covariâncias diferentes** — é essa
diferença que dá vantagem ao QDA sobre o LDA.

$$X\mid Y=0 \sim N\!\left(\begin{bmatrix}0\\0\end{bmatrix},
   \begin{bmatrix}1{,}00 & 0{,}75\\ 0{,}75 & 1{,}00\end{bmatrix}\right),
  \qquad
  X\mid Y=1 \sim N\!\left(\begin{bmatrix}2{,}20\\0{,}77\end{bmatrix},
   \begin{bmatrix}1{,}60 & -0{,}85\\ -0{,}85 & 0{,}70\end{bmatrix}\right).$$

Como conhecemos as densidades, o classificador de Bayes é imediato: com prioris
iguais, ele decide por $Y=1$ quando $f_1(x) > f_0(x)$.

In [ ]:
S0 = np.array([[1.0, 0.75], [0.75, 1.0]])
S1 = np.array([[1.6, -0.85], [-0.85, 0.7]])
mu0 = np.array([0.0, 0.0])
mu1 = np.array([2.2, 2.2 * 0.35])


def gera(n, rng):
    n0 = n // 2
    X = np.vstack([rng.multivariate_normal(mu0, S0, size=n0),
                   rng.multivariate_normal(mu1, S1, size=n - n0)])
    y = np.r_[np.zeros(n0), np.ones(n - n0)].astype(int)
    return X, y


rng = np.random.default_rng(2026)
X_grande, y_grande = gera(200000, rng)

f0 = multivariate_normal(mu0, S0).pdf(X_grande)
f1 = multivariate_normal(..., ...).pdf(X_grande)                 # (a) e (b)

decisao_bayes = (...).astype(int)                             # (c)
erro_bayes = np.mean(decisao_bayes != y_grande)

print(f"erro de Bayes = {erro_bayes:.4f}   (acuracia maxima = {1 - erro_bayes:.4f})")

> **Sua vez.** Desenhe as duas nuvens (500 pontos) num diagrama de dispersão, com
> cores diferentes por classe. Dá para ver por que a fronteira ótima não é uma
> reta?

---
## Exercício 2 — quatro classificadores, uma população

Agora esqueça que conhecemos as densidades e estime tudo a partir de 200
observações. Compare os quatro classificadores da aula contra o teto do
Exercício 1.

In [ ]:
rng = np.random.default_rng(2026)
X_tr, y_tr = gera(200, rng)
X_te, y_te = gera(20000, rng)

classificadores = [
    ("LDA",        ...),                 # (a)
    ("QDA",        ...),              # (b)
    ("GaussianNB", GaussianNB()),
    ("logistica",  LogisticRegression()),
]

for nome, modelo in classificadores:
    acuracia = modelo.fit(X_tr, y_tr).score(..., ...)         # (c) e (d)
    print(f"{nome:12s} acuracia de teste {acuracia:.4f}")

# o teto, recalculado no mesmo conjunto de teste
otimo = (multivariate_normal(mu1, S1).pdf(X_te)
         > multivariate_normal(mu0, S0).pdf(X_te)).astype(int)
print(f"{'Bayes':12s} acuracia de teste {np.mean(otimo == y_te):.4f}")

---
## Exercício 3 — LDA contra QDA, em função de $n$ e de $p$

A Lista Teórica 08 contou os parâmetros: em $p=2$ o QDA custa 3 a mais que o LDA;
em $p=10$, custa 55 a mais. Vamos ver o efeito disso.

A população em $\mathbb{R}^p$ é a mesma ideia: duas gaussianas com covariâncias fixas
diferentes, e médias separadas nas três primeiras coordenadas.

In [ ]:
_COVS = {}


def covariancia_fixa(d, qual):
    """Mesma matriz em toda chamada, para a populacao nao mudar entre repeticoes."""
    if (d, qual) not in _COVS:
        g = np.random.default_rng(1000 + qual)
        A = g.normal(size=(d, d))
        _COVS[(d, qual)] = (A @ A.T) / d + np.eye(d) * 0.5
    return _COVS[(d, qual)]


def gera_d(n, d, rng):
    n0 = n // 2
    mu = np.zeros(d)
    mu[:min(3, d)] = [1.6, 1.0, 0.7][:min(3, d)]
    X = np.vstack([rng.multivariate_normal(np.zeros(d), covariancia_fixa(d, 0), size=n0),
                   rng.multivariate_normal(mu, covariancia_fixa(d, 1), size=n - n0)])
    y = np.r_[np.zeros(n0), np.ones(n - n0)].astype(int)
    return X, y

In [ ]:
ns = np.array([20, 30, 50, 100, 300, 1000])

for d in (2, 10):
    X_teste, y_teste = gera_d(20000, d, np.random.default_rng(99))
    rng = np.random.default_rng(2026)
    media_lda, media_qda = [], []

    for n in ns:
        acc_lda, acc_qda = [], []
        for _ in range(60):
            X, y = gera_d(int(n), d, rng)
            # com n pequeno e d grande, a covariancia do QDA pode ser singular
            for Modelo, acc in ((LinearDiscriminantAnalysis, acc_lda),
                                (QuadraticDiscriminantAnalysis, acc_qda)):
                try:
                    acc.append(Modelo().fit(X, y).score(X_teste, y_teste))
                except Exception:
                    acc.append(np.nan)
        media_lda.append(...)                     # (a)
        media_qda.append(np.nanmean(acc_qda))

    media_lda, media_qda = np.array(media_lda), np.array(media_qda)
    virada = ns[...]            # (b) primeiro n em que o QDA passa

    print(f"d={d}:")
    print(f"   n   = {ns.tolist()}")
    print(f"   LDA = {media_lda.round(4).tolist()}")
    print(f"   QDA = {media_qda.round(4).tolist()}")
    print(f"   QDA passa a ganhar em n = {virada}")

---
## Exercício 4 — e num banco de verdade?

Nos exercícios anteriores nós geramos os dados, então sabíamos qual modelo estava
certo. No `breast_cancer` — 569 tumores, 30 medidas de imagem, resposta binária
benigno/maligno — ninguém sabe.

In [ ]:
dados = load_breast_cancer()
X_bc, y_bc = dados.data, dados.target

print(f"n = {X_bc.shape[0]}, d = {X_bc.shape[1]}, "
      f"positivos = {100 * y_bc.mean():.1f}%")

cv = skm.StratifiedKFold(..., shuffle=True, random_state=2026)      # (a)

modelos = [
    ("LDA",        LinearDiscriminantAnalysis()),
    ("QDA",        QuadraticDiscriminantAnalysis()),
    ("GaussianNB", GaussianNB()),
    ("logistica",  Pipeline([("escala", ...),        # (b)
                             ("modelo", LogisticRegression(max_iter=5000))])),
]

for nome, modelo in modelos:
    acuracia = skm.cross_val_score(modelo, X_bc, y_bc, cv=...).mean()   # (c)
    print(f"{nome:12s} acuracia (CV) {acuracia:.4f}")

> **Sua vez.** A acurácia aqui é uma métrica razoável porque as classes estão
> quase equilibradas (62,7% contra 37,3%). Rode de novo com
> `scoring="roc_auc"`. A ordem entre os quatro muda?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | o erro de Bayes desta população é 0,0958 — o teto de acurácia é 0,9042 |
| 2 | o QDA **alcança o teto** com $n=200$ (0,9025 contra 0,9023): ele é o modelo correto |
| 2 | a logística bate o LDA (0,8716 contra 0,8627), mesmo os dois traçando retas |
| 3 | em $p=2$ a escolha LDA/QDA muda menos de 1 ponto; em $p=10$ muda **16 pontos** |
| 4 | no `breast_cancer` a ordem se inverte: a logística ganha e o QDA cai para terceiro |

**A seguir.** A Aula 08 pega a segunda metade do problema. Todos os números desta
lista foram acurácias, e a população estava equilibrada. Quando 10% das
observações são positivas, a acurácia deixa de dizer qualquer coisa útil.